In [1]:
%load_ext autoreload
%autoreload 2
import torch
from utils import goto_project_root
from utils.path_settings import MODEL_SAVE_PATH, DATA_PATH, LOG_PATH, CONFIG_PATH, OBJECT_PATH
from torch.utils.tensorboard import SummaryWriter
import SimulateDatasets.GenTrainingData as g
from utils import create_splits, get_dataloaders, force_remove_dir
from Network_models import Trainer as t
from importlib import reload
import os
reload(t)
reload(g)
import time
import numpy as np
from pytorch3d.transforms import so3_relative_angle
from pytorch3d.transforms import quaternion_to_matrix
import json
import Train_task.train_from_config_names as train
reload(train)
import torchvision.models as models
from torchvision.models import ConvNeXt_Tiny_Weights

In [5]:
MODEL_SAVE_PATH + "\\FC_"

'D:\\Projects\\mental-rotations\\models\\FC_'

In [7]:
network_name = "ConvNeXt_Tiny"
FC_size = 16 # This is the input size for the RNN part of the RNN structure (we will be loading FC_RNN).
rnn_hidden_size = 8
task_RNN_pretrained = "0.1q"
rnn_cell_type = "GRU"
rnn_special_name = "constant"
distance_loss = "geodesic_gradual"
seq_len = 100
prep_phase = 50
task_name = "1.1"
re_train = 0

config = {"model_specs": {
    "model_name": "Imported_CNN_RNN",  # to be added (CustomRNN or FC_RNN)
    "model_path": "Network_models.CNN_models",
    "model_params": {
        "convnet": "ConvNeXt_Tiny",
        "weight_string": "IMAGENET1K_V1", 
        "rnn_input_size": FC_size,
        "rnn": {
            "freeze_rnn": False, 
            "model_name": "FC_RNN",
            "model_path": "Network_models.RNN_models",
            "remove_FC": True, 
            "saved_state_dict_path": None,
            "model_params": {
                "input_size": 8,
                "hidden_size": 8, # to be changed
                "num_layers": 1, # to be changed
                "output_size": 3,
                "FC_dim": FC_size,
                "cell_type": rnn_cell_type,  # to be added
            }, 
            "stepwise": False, 
            "saved_model_path": MODEL_SAVE_PATH + 
                                f"\\FC_{FC_size}_{rnn_cell_type}_{1}layer_{rnn_hidden_size}hidden_{task_RNN_pretrained}"
                                f"_model_{'gradual' if distance_loss[-7:] == 'gradual' else ''}"
                                f"{'_' + rnn_special_name if rnn_special_name is not None else ''}\\best_model.pth",
        }, 
    }
}, "model_id": network_name, "device": "cuda", "scheduler": True, # Uses the ReduceLROnPlateau scheduler
    "optimizer_specs": {
    "optimizer_name": "Adam",
    "optimizer_params": {
        "lr": 0.01
    }
}, "distance_loss": distance_loss, "gradual_loss_weighting": "constant+linear", "regularisation_loss": "L2",
    "distance_weight": 1, "output_regs_weight": 1, "silence_activity": False,
    # silence_activity is the knob for suppressing activity in the silence phase
    "training_config": {
        "task_id": "1.1",
        "batch_size": 1024,
        "mini_batch_size": 128,
        "seq_len": seq_len,
        "prep_phase": prep_phase,
        "resolution": 512,
        "object_path": OBJECT_PATH + "\\cow_mesh\\cow.obj"
    }, 'save_path': MODEL_SAVE_PATH + f"\\{network_name}_{task_name}_model", 'log_path': LOG_PATH,
    'check_path': MODEL_SAVE_PATH + f"\\{network_name}_model_checkpoints"}
config['training_config']['data_save_path'] = DATA_PATH + f"\\Data_{task_name}_res{config['training_config']['resolution']}.pth"
    
                                                                                                              

if config['distance_loss'][-7:] == "gradual":
    config['save_path'] = MODEL_SAVE_PATH + (f"\\{network_name}_{task_name}_res{config['training_config']['resolution']}"
                                             f"_model_gradual")
    config['check_path'] = (MODEL_SAVE_PATH + 
                            f"\\{network_name}_res{config['training_config']['resolution']}_model_checkpoints_gradual")
    config['log_path'] = LOG_PATH + (f"\\{network_name}_{task_name}_res{config['training_config']['resolution']}"
                                     f"_model_gradual")
    config['model_specs']['model_params']['stepwise'] = True
else:
    config['model_specs']['model_params']['stepwise'] = False

for path in [config['save_path'], config['log_path'], config['check_path']]:
    if not os.path.exists(path):
        os.makedirs(path)
    elif not re_train:
        print(f"Path already exists for {network_name} in task {task_name}. Assume the model is already trained.")
    else: # re_train
        force_remove_dir(path)
        os.makedirs(path)
print(config['save_path'])
json.dump(config, open(CONFIG_PATH + f"\\{network_name}_{task_name}_configs.json", "w"))

Path already exists for ConvNeXt_Tiny in task 1.1. Assume the model is already trained.
Path already exists for ConvNeXt_Tiny in task 1.1. Assume the model is already trained.
Path already exists for ConvNeXt_Tiny in task 1.1. Assume the model is already trained.


In [3]:
experimental_trainer = t.Trainer(config)
experimental_trainer.model.resolution = config['training_config']['resolution']

C:\Users\timmy\anaconda3\envs\mental-rotations\lib\site-packages\torchvision\models\_utils.py:135: UserWarning: Using 'weights' as positional parameter(s) is deprecated since 0.13 and may be removed in the future. Please use keyword parameter(s) instead.
  warnings.warn(


Using a scheduler


In [4]:
reload(g)
if not os.path.exists(config['training_config']['data_save_path']):
    print('Generating data')
    data = g.gen_training_data(config['training_config'])
dataloaders = get_dataloaders(data, batch_size = config['training_config']['mini_batch_size'], k = 5)

An exception occurred in telemetry logging.Disabling telemetry to prevent further exceptions.
Traceback (most recent call last):
  File "C:\Users\timmy\anaconda3\envs\mental-rotations\lib\site-packages\iopath\common\file_io.py", line 946, in __log_tmetry_keys
    handler.log_event()
  File "C:\Users\timmy\anaconda3\envs\mental-rotations\lib\site-packages\iopath\common\event_logger.py", line 97, in log_event
    del self._evt
AttributeError: _evt


Generating data


In [6]:
config['save_path']

'D:\\Projects\\mental-rotations\\models\\ConvNeXt_Tiny_1.1_res512_model_gradual'

In [5]:
epochs = 100
print(f"Training {network_name} on task {task_name}, saving to {config['save_path']}; copy the below for logs")
print(f"tensorboard --logdir={config['log_path']}")
for i in range(len(dataloaders)):
    sub_log_path = config["log_path"] + f"\\split_{i + 1}"
    sub_check_path = config["check_path"] + f"\\split_{i + 1}"

    force_remove_dir(sub_log_path) # Probably redundant but helps to ensure no old logs are kept
    os.makedirs(sub_log_path, exist_ok = True)
    os.makedirs(sub_check_path, exist_ok = True)
   
for i, (train_loader, val_loader) in enumerate(dataloaders):
    print(f"Training split {i + 1}")
    experimental_trainer.refresh()
    sub_log_path = config["log_path"] + f"\\split_{i + 1}"
    sub_check_path = config["check_path"] + f"\\split_{i + 1}"

    experimental_trainer.train(train_loader,
                      val_loader,
                      epochs=epochs,
                      save_path = config["save_path"] + f"\\model_{i+1}.pth",
                      check_path= sub_check_path,
                      log_path = sub_log_path)

    experimental_trainer.save_model(config["save_path"] + f"\\model_{i+1}.pth", full=1)
experimental_trainer.load_best_model()
print(f"Training {network_name} on task {task_name} complete. Saving best model to {config['save_path']}.")
experimental_trainer.save_model(config["save_path"] + f"\\best_model.pth", full=1)


Training ConvNeXt_Tiny on task 1.1, saving to D:\Projects\mental-rotations\models\ConvNeXt_Tiny_1.1_res512_model_gradual; copy the below for logs
tensorboard --logdir=D:\Projects\mental-rotations\logs\ConvNeXt_Tiny_1.1_res512_model_gradual
Directory does not exist: D:\Projects\mental-rotations\logs\ConvNeXt_Tiny_1.1_res512_model_gradual\split_1
Directory does not exist: D:\Projects\mental-rotations\logs\ConvNeXt_Tiny_1.1_res512_model_gradual\split_2
Directory does not exist: D:\Projects\mental-rotations\logs\ConvNeXt_Tiny_1.1_res512_model_gradual\split_3
Directory does not exist: D:\Projects\mental-rotations\logs\ConvNeXt_Tiny_1.1_res512_model_gradual\split_4
Directory does not exist: D:\Projects\mental-rotations\logs\ConvNeXt_Tiny_1.1_res512_model_gradual\split_5
Training split 1


KeyboardInterrupt: 

In [80]:
task = "1.1"
network_best_split = {
    "ConvNeXt_Tiny": 1,
}
for network_name, split in network_best_split.items():
    epochs = 250
    trainer = t.Trainer(config)
    trainer.load_checkpoint(config['check_path'] + f"\\split_{split}\\checkpoint99.pth")
    # trainer.scheduler = None
    if not os.path.exists(config['training_config']['data_save_path']):
        print('Generating data')
        data = g.gen_training_data(config['training_config'])
    import utils as u
    reload(u)
    dataloaders = u.get_dataloaders(data, config['training_config']['mini_batch_size'], 1)
    print("Continue training the best model")
    sub_log_path = config['log_path'] + "\\continue_training"
    sub_check_path = config['check_path'] + "\\continue_training"
    force_remove_dir(sub_log_path)
    force_remove_dir(sub_check_path)
    os.makedirs(sub_log_path, exist_ok=True)
    os.makedirs(sub_check_path, exist_ok=True)
    trainer.train(
        dataloaders[0][0],
        dataloaders[0][1],
        epochs=epochs,
        save_path = config['save_path'] + "\\continue_training.pth",
        check_path=sub_check_path,
        log_path = sub_log_path,
    )
    trainer.save_model(config['save_path'] + "\\continue_training.pth", full = 1)
    trainer.load_best_model()
    trainer.save_model(config['save_path'] + "\\best_model.pth", full = 1)

Using a scheduler
Continue training the best model
Successfully removed directory: D:\Projects\mental-rotations\logs\ConvNeXt_Tiny_1.1_res256_model_gradual\continue_training
Successfully removed directory: D:\Projects\mental-rotations\models\ConvNeXt_Tiny_res256_model_checkpoints_gradual\continue_training


In [44]:
for i, layer in enumerate(experimental_trainer.model.children()):
    if hasattr(layer, "frozen"): 
        print(f"Layer {i} is frozen: {layer.frozen}")
    print(f"Layer {i} is {layer}")
experimental_trainer.refresh()

Layer 0 is frozen: True
Layer 0 is Sequential(
  (0): Conv2dNormActivation(
    (0): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
    (1): LayerNorm2d((96,), eps=1e-06, elementwise_affine=True)
  )
  (1): Sequential(
    (0): CNBlock(
      (block): Sequential(
        (0): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96)
        (1): Permute()
        (2): LayerNorm((96,), eps=1e-06, elementwise_affine=True)
        (3): Linear(in_features=96, out_features=384, bias=True)
        (4): GELU(approximate='none')
        (5): Linear(in_features=384, out_features=96, bias=True)
        (6): Permute()
      )
      (stochastic_depth): StochasticDepth(p=0.0, mode=row)
    )
    (1): CNBlock(
      (block): Sequential(
        (0): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96)
        (1): Permute()
        (2): LayerNorm((96,), eps=1e-06, elementwise_affine=True)
        (3): Linear(in_features=96, out_features=384, bias=True)
  

In [57]:
experimental_trainer.load_model(config["save_path"] + f"\\best_model.pth", full=1)
experimental_trainer.model.rotate(torch.tensor([1, 0, 0, 0], dtype = torch.get_default_dtype(), device = experimental_trainer.device))

An exception occurred in telemetry logging.Disabling telemetry to prevent further exceptions.
Traceback (most recent call last):
  File "C:\Users\timmy\anaconda3\envs\mental-rotations\lib\site-packages\iopath\common\file_io.py", line 946, in __log_tmetry_keys
    handler.log_event()
  File "C:\Users\timmy\anaconda3\envs\mental-rotations\lib\site-packages\iopath\common\event_logger.py", line 97, in log_event
    del self._evt
AttributeError: _evt


torch.Size([3, 128, 128])
torch.Size([1, 3, 128, 128])


tensor([[ 0.8746,  0.4008, -0.0661,  0.2648],
        [ 0.9292,  0.2858, -0.0508,  0.2288],
        [ 0.9901,  0.0983, -0.0273,  0.0963],
        [ 0.9996,  0.0110,  0.0011,  0.0244],
        [ 0.9996, -0.0227,  0.0089, -0.0149],
        [ 0.9990, -0.0321,  0.0067, -0.0298],
        [ 0.9988, -0.0342,  0.0042, -0.0342],
        [ 0.9988, -0.0340,  0.0041, -0.0354],
        [ 0.9988, -0.0331,  0.0051, -0.0358],
        [ 0.9988, -0.0324,  0.0060, -0.0358],
        [ 0.9988, -0.0319,  0.0066, -0.0356],
        [ 0.9989, -0.0316,  0.0069, -0.0353],
        [ 0.9989, -0.0315,  0.0070, -0.0352],
        [ 0.9989, -0.0314,  0.0071, -0.0351],
        [ 0.9989, -0.0313,  0.0071, -0.0350],
        [ 0.9989, -0.0313,  0.0071, -0.0350],
        [ 0.9989, -0.0313,  0.0071, -0.0350],
        [ 0.9989, -0.0313,  0.0071, -0.0350],
        [ 0.9989, -0.0313,  0.0071, -0.0350],
        [ 0.9989, -0.0313,  0.0071, -0.0350],
        [ 0.9989, -0.0313,  0.0071, -0.0350],
        [ 0.9989, -0.0313,  0.0071

In [60]:
experimental_trainer.save_model(config['save_path'] + "\\not_trained.pth", full = 1)

In [61]:
experimental_trainer.model.train()

Imported_CNN_RNN(
  (conv): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
      (1): LayerNorm2d((96,), eps=1e-06, elementwise_affine=True)
    )
    (1): Sequential(
      (0): CNBlock(
        (block): Sequential(
          (0): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96)
          (1): Permute()
          (2): LayerNorm((96,), eps=1e-06, elementwise_affine=True)
          (3): Linear(in_features=96, out_features=384, bias=True)
          (4): GELU(approximate='none')
          (5): Linear(in_features=384, out_features=96, bias=True)
          (6): Permute()
        )
        (stochastic_depth): StochasticDepth(p=0.0, mode=row)
      )
      (1): CNBlock(
        (block): Sequential(
          (0): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96)
          (1): Permute()
          (2): LayerNorm((96,), eps=1e-06, elementwise_affine=True)
          (3): Linear(in_featu